# Task B Run 25 -- Gemma-4-12B TAPT before classification

Run 21 established a strong Gemma-4-12B QLoRA recipe: the two-seed model reached 0.6792 macro-F1 on the fixed 530-row holdout. This run tests one additional idea only: continued pretraining on the Task B comment style before supervised classification.

**Stage 1:** each seed performs one causal-language-model TAPT epoch on the 3,002 classifier-training comments. The 530 holdout comments are not used for TAPT.

**Stage 2:** the saved TAPT adapter is loaded into the existing Gemma classifier and trained with the same 3-epoch, rank-16, attention-plus-MLP QLoRA recipe as Run 21.

The two TAPT-seed probability matrices are averaged and compared with Run 21's saved no-TAPT Gemma probabilities using a paired bootstrap on exactly the same rows. This notebook is evaluation-only and does not create a submission ZIP; a full-data fit should happen only if TAPT wins.

## Before running

- Add Run 21's notebook output with **Add Input -> Notebook Output**; its no-TAPT probabilities are the control.
- Use **GPU T4 x2**, Internet enabled, and the `HF_TOKEN` Kaggle secret.
- Save and run all. Expect roughly 6--10 hours; causal-LM TAPT is more expensive than classification-only QLoRA.

In [ ]:
import glob, json, os, pathlib, shutil, subprocess, sys, time

REPO = "https://github.com/robinpnalex/Hastika-ICON2026.git"
BRANCH = "task-b"
WORK = "/kaggle/working/hastika"

def git(*args):
    return subprocess.run(["git", "-C", WORK, *args], check=True, capture_output=True, text=True).stdout.strip()

try:
    remote = subprocess.run(["git", "ls-remote", "--exit-code", REPO, f"refs/heads/{BRANCH}"], check=True, capture_output=True, text=True, timeout=60).stdout.split()[0]
except Exception as e:
    raise SystemExit("Cannot reach GitHub. Turn on Kaggle Internet and restart the session.") from e
if os.path.isdir(WORK + "/.git"):
    git("fetch", "-q", "--depth", "1", "origin", BRANCH)
    git("reset", "-q", "--hard", "FETCH_HEAD")
else:
    subprocess.run(["git", "clone", "-q", "-b", BRANCH, "--depth", "1", REPO, WORK], check=True)
head = git("rev-parse", "HEAD")
assert head == remote, f"clone is at {head[:8]} but {BRANCH} is at {remote[:8]}"
os.chdir(WORK)
os.environ["PYTHONPATH"] = os.path.join(WORK, "src")
sys.path.insert(0, os.path.join(WORK, "src"))

try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    print("HF_TOKEN secret loaded")
except Exception:
    print("HF_TOKEN not found; Gemma access may fail")

# Install the repo-level preprocessing dependencies before importing hastika.*.
# Kaggle's base image does not include ftfy, which preprocessing.py imports.
subprocess.run('pip install -q emoji ftfy sentencepiece protobuf "transformers>=4.45,<6"', shell=True, check=True)

import numpy as np
import pandas as pd
import torch
assert torch.cuda.is_available(), "select a GPU accelerator"
N_GPU = torch.cuda.device_count()
print("repo", git("log", "-1", "--oneline"))
print("gpus:", [torch.cuda.get_device_name(i) for i in range(N_GPU)])

OUT = pathlib.Path("/kaggle/working/b25_gemma_tapt_outputs")
TAPT_RUNS, CLS_RUNS, LOGS = OUT / "tapt", OUT / "classifier", OUT / "logs"
for p in (TAPT_RUNS, CLS_RUNS, LOGS): p.mkdir(parents=True, exist_ok=True)
HF_CACHE = "/tmp/hf"
os.environ["HF_HUB_CACHE"] = HF_CACHE

from hastika.common.gpu_queue import run_queue
from hastika.task_b.combined_holdout import LABELS, fingerprint, paired_bootstrap, report
SPLIT = pathlib.Path(WORK) / "data/derived/task_b_combined_holdout"
train = pd.read_csv(SPLIT / "train.csv")
holdout = pd.read_csv(SPLIT / "holdout.csv")
assert (len(train), len(holdout)) == (3002, 530), (len(train), len(holdout))
HOLDOUT_FP = fingerprint(holdout["id"])
assert HOLDOUT_FP == "f85f4f049b", HOLDOUT_FP
y_ho = holdout["Hate Category"].map(LABELS.index).to_numpy()
MODEL = "google/gemma-4-12B"
S = "gemma-4-12b"
SEEDS = (42, 43)
print("train", len(train), "holdout", len(holdout), "fingerprint", HOLDOUT_FP)

In [ ]:
LLM_PY = "/tmp/llmenv/bin/python"
if not os.path.exists(LLM_PY):
    subprocess.run([sys.executable, "-m", "venv", "--system-site-packages", "--without-pip", "/tmp/llmenv"], check=True)
subprocess.run(f'{LLM_PY} -m pip install -q "transformers>=5.12,<6" "peft>=0.17" "bitsandbytes>=0.46" "accelerate>=1.0" emoji ftfy sentencepiece protobuf', shell=True, check=True)
subprocess.run([LLM_PY, "-c", "import torch, transformers, peft, bitsandbytes; print(torch.__version__, transformers.__version__, peft.__version__, bitsandbytes.__version__)"], check=True)

from huggingface_hub import snapshot_download
snapshot_download(MODEL, cache_dir=HF_CACHE, token=os.environ.get("HF_TOKEN") or None, allow_patterns=["*.json", "*.safetensors", "*.model", "*.txt", "tokenizer*", "*.jinja"])
print("Gemma cached; free /tmp", f"{shutil.disk_usage('/tmp').free / 2**30:.0f} GB")

def tapt_job(seed):
    name = f"{S}_tapt_s{seed}"
    return {"name": name, "log": str(LOGS / f"{name}.log"),
            "cmd": (f"{LLM_PY} -u -m hastika.task_b.gemma_tapt --model {MODEL} "
                    f"--train {SPLIT / 'train.csv'} --out {TAPT_RUNS / name} "
                    f"--seed {seed} --epochs 1 --r 16 --bs 1 --grad-accum 8 --max-len 256 --cache {HF_CACHE}")}

def classifier_job(seed):
    name = f"{S}_tapt_cls_s{seed}"
    adapter = TAPT_RUNS / f"{S}_tapt_s{seed}" / "adapter"
    return {"name": name, "log": str(LOGS / f"{name}.log"),
            "cmd": (f"{LLM_PY} -u -m hastika.task_b.llm_classifier --model {MODEL} "
                    f"--train {SPLIT / 'train.csv'} --eval {SPLIT / 'holdout.csv'} "
                    f"--predict {SPLIT / 'holdout.csv'} --out {CLS_RUNS / name} "
                    f"--seed {seed} --epochs 3 --r 16 --init-adapter {adapter} "
                    f"--cache {HF_CACHE}")}

tapt_jobs = [tapt_job(s) for s in SEEDS]
print("TAPT jobs:", [j["name"] for j in tapt_jobs])

## Stage 1: causal-language-model TAPT

Each Gemma copy sees only the 3,002 training comments. It learns to predict the next token with a rank-16 QLoRA adapter. The saved adapter is then reused by the classifier stage.

In [ ]:
T_START = time.time()
HARD_STOP = T_START + 11 * 3600
tapt_codes = run_queue(tapt_jobs, N_GPU, raise_on_fail=False, stop_at=HARD_STOP)
print("TAPT exit codes:", tapt_codes)
for s in SEEDS:
    m = TAPT_RUNS / f"{S}_tapt_s{s}" / "metrics.json"
    if m.exists(): print(s, json.load(open(m))["history"])

## Stage 2: the existing Gemma classifier recipe

This keeps the supervised stage unchanged: balanced class weights, label smoothing, rank 16, attention-plus-MLP targets, and three epochs. The only new input is the TAPT adapter.

In [ ]:
ready = [s for s in SEEDS if (TAPT_RUNS / f"{S}_tapt_s{s}" / "adapter").exists()]
assert ready, "no TAPT adapter finished; inspect the TAPT logs"
cls_jobs = [classifier_job(s) for s in ready]
cls_codes = run_queue(cls_jobs, N_GPU, raise_on_fail=False, stop_at=HARD_STOP)
print("classifier exit codes:", cls_codes)
print(f"elapsed {(time.time() - T_START) / 3600:.1f} h")

In [ ]:
def load_probs(path):
    p = pathlib.Path(path) / "holdout_probs.npy"
    return np.load(p) if p.exists() else None

found = sorted(glob.glob("/kaggle/input/**/b21_outputs/result.json", recursive=True))
assert found, "attach Run 21 notebook output so the no-TAPT control can be loaded"
B21 = pathlib.Path(found[0]).parent
r21 = json.load(open(B21 / "result.json"))
assert r21["holdout_fingerprint"] == HOLDOUT_FP
baseline = {s: load_probs(B21 / "runs" / f"{S}_ho_s{s}") for s in SEEDS}
baseline = {s: p for s, p in baseline.items() if p is not None}
tapt = {s: load_probs(CLS_RUNS / f"{S}_tapt_cls_s{s}") for s in ready}
tapt = {s: p for s, p in tapt.items() if p is not None}
assert baseline and tapt, "missing baseline or TAPT probabilities"
for s, p in baseline.items(): report(y_ho, p, f"no TAPT seed {s}")
for s, p in tapt.items(): report(y_ho, p, f"TAPT seed {s}")
baseline_mean = np.mean(list(baseline.values()), 0)
tapt_mean = np.mean(list(tapt.values()), 0)
base_score = report(y_ho, baseline_mean, "no TAPT mean")
tapt_score = report(y_ho, tapt_mean, "TAPT mean")
comparison = paired_bootstrap(y_ho, tapt_mean, baseline_mean, n=5000, seed=2500)
print(f"TAPT - no TAPT: {comparison['diff']:+.4f}; CI [{comparison['ci95'][0]:+.4f}, {comparison['ci95'][1]:+.4f}]; P(better) {comparison['p_better']:.2f}")

In [ ]:
record = {
    "commit": head, "model": MODEL, "holdout_rows": len(holdout),
    "holdout_fingerprint": HOLDOUT_FP, "tapt_rows": len(train),
    "tapt_epochs": 1, "classifier_epochs": 3, "seeds": list(SEEDS),
    "no_tapt_macro_f1": base_score, "tapt_macro_f1": tapt_score,
    "tapt_minus_no_tapt": comparison, "tapt_exit_codes": tapt_codes,
    "classifier_exit_codes": cls_codes, "completed_tapt_seeds": sorted(map(int, tapt)),
}
json.dump(record, open(OUT / "result.json", "w"), indent=2)
print(json.dumps(record, indent=2))
print("No submission ZIP is created; only promote TAPT to a full-data fit if this holdout comparison is convincing.")